# Plot slip inversion of the synthetic GNSS displacement at Nicoya (Costa Rica) within a heterogeneous half-space, compared with a homogeneous solution

* Can deal with ground-truth slip of various pattern. In particular, checkerboard pattern. 

* The max amplitude is to simulate either the max locking (== the long-term trench-normal velocity, <100 mm/yr), or the coseismic (say a few m)

* The goal is to get a sense of how the recovery will change under the heterogeneity, i.e., can you say anything at offshore if you see something with heterogeneity?

In [ ]:
# Load libraries
import numpy as np
import pygmt
import matplotlib.pyplot as plt
import pandas as pd
import utils_plot as utp
import meshio

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define folder to save the figures
figpath = "/home/staff/chao/SSEinv/Nicoya/figures/synth/"
os.makedirs(figpath, exist_ok=True)

# flag to indicate whether to save figures
# flag_savefig = True
flag_savefig = False

# Define the inversion results path
resultpath = "syn_slip/"

# define mesh name
# meshname = "nicoya"  # larger fault interface
# meshname = "nicoya2"   # This has a smaller fault interface
# meshname = "nicoyaCK"   # local interface model from C. Kyriakopoulos_etal2015JGRSE
# meshname = "nicoyaCK2"   # same as above but 5-km mesh size on fault
# meshname = "nicoyaCK3"   # fault zone extended to the whole subduction zone
meshname = "nicoyaCK4"   # same as CK2, but connecting the trench nowprint(meshname)
print(meshname)

# # Read the plate interface file
# plate_file = "plateinterface/nicoya_slab2_100_10_10.txt"
# df_plate = utp.parse_plate_interface_file(datadir + plate_file)
# depths = np.array(df_plate['dep'].unique())
# print("depths:", depths)

# Read the mesh file for generating the slab interface
mesh_file = "Kyriakopoulos2016JGR/Nicoya_interface.e"
mesh = meshio.read(datadir + mesh_file)
points = mesh.points  # shape (n_points, 3)
df_plate = pd.DataFrame(points, columns=["x", "y", "z"])
# define the centroid of relative coordinates that is consistent with mesh
lon0, lat0 = -84, 7     # from Christos's email
# convert to relative locations in meters, and then rotate
rot = 45  # rotation angle in degrees, positive is CCW
df_plate['lon'], df_plate['lat'] = utp.ckm2LLd(df_plate['x'], df_plate['y'], lon0, lat0, -rot)
df_plate['z'] = df_plate['z'] /1e3  
# print(df_plate.head())

# Read the trench file
# trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_geo.txt"
# trench = pd.read_csv(trench_file, sep=r'\s+', names=['lon', 'lat'])
trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_xyz.txt"
trench = pd.read_csv(trench_file, sep=r'\s+', names=['x', 'y'])
trench['lon'], trench['lat'] = utp.ckm2LLd(trench['x'], trench['y'], lon0, lat0, -rot)
# print(trench.head())

# anaysis type, locking or coseismic?
type = "locking"  # "locking" or "coseismic"
# type = "coseismic"

# whethere the synthetics are polluted
pollute = True  # True or False
# pollute = False
print(pollute)

# noise std type, either 'uniform' or 'datastd'
pollute_type = 'uniform'  # uniform noise for all stations
# pollute_type = 'datastd'  # use the data standard deviation as noise std
print(pollute_type)

m2mm = 1e3  # meter to mm
km2m = 1e3   # km to m

if type == "locking":
       # read in the lon and lat of stations
       # obs_file = "data/Feng_etal_JGR_2012table1.csv" # original data from Feng et al. 2012
       obs_file = "data/Kyriakopoulos_novolcano.csv"    # same as above, but with volcano sites removed

       # note that the height is in m, Dt in years, the original displacement data is in mm/yr
       # the disp relative to the stable Caribbean plate will be used in the inversion
       # From ACOS to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
       df = pd.read_csv(datadir + obs_file, sep=",", skiprows=1, \
                     names=['site', 'lat', 'lon', 'height', 'Dt', 'Days', \
                            'vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                            'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car'])
       df['lon'] = -1*df['lon'] # convert to east positive, as the original data is west positive
       # Convert mm to m, needed for inversion
       cols_to_convert = ['vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                     'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car']
       df[cols_to_convert] = df[cols_to_convert] / m2mm  # Convert mm to m

       # displacement noise standard deviations, in m 
       if pollute:
              if pollute_type == 'uniform':
                     noise_std_h = 0.5 * (df['vx_std_Car'].mean() + df['vy_std_Car'].mean()) 
                     noise_std_v = df['vz_std_Car'].mean()
                     error_e, error_n, error_v  = noise_std_h, noise_std_h, noise_std_v
                     
              elif pollute_type == 'datastd':
                     error_e, error_n, error_v = df['vx_std_Car'], df['vy_std_Car'], df['vz_std_Car']
                     
       else:
              error_e, error_n, error_v = df['vx_std_Car']*0, df['vy_std_Car']*0, df['vz_std_Car']*0

else:
       obs_file = "data/Protti_etal_2014_tableS1.csv"
       # note that the height is in m, duration and dates are in yr, and the displacements and errors are already in m
       # From BAGA to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
       df = pd.read_csv(datadir + obs_file, sep=",", skiprows=1, \
                     names=['site', 'lon', 'lat', 'elv', 'ux', 'uy', 'uz', \
                            'ux_std', 'uy_std', 'uz_std', 'date0', 'date1', 'duration'])
       
       # displacement noise standard deviations
       if pollute:
              noise_std_h = 0.5 * (df['ux_std'].mean() + df['uy_std'].mean())
              error_e, error_n, error_v  = noise_std_h, noise_std_h, noise_std_h
       else:
              error_e, error_n, error_v = df['ux_std'], df['uy_std'], df['uz_std']

# print(df.head())  # Preview the data
print(df['lon'].min(), df['lon'].max(), df['lat'].min(), df['lat'].max())
print("Displacement noise std: ", error_e.mean(), error_n.mean(), error_v.mean())

In [ ]:
# flag to indicate if the slip inversion was bounded
BOUNDED = True
# BOUNDED = False

BOUND_TYPE = 'both'

# choose function type,  available choices are 'tanh'- Hyperbolic tangent (default), 'arctan' - Arctangent scaled by 2/π  
# 'sigmoid' - 2/(1+exp(-x)) - 1, and 'sqrt' - x/sqrt(1+x²)
FUNCTION_TYPE = 'tanh'
# FUNCTION_TYPE = 'sigmoid'

if BOUNDED:
    # Define slip bounds based on your problem
    V_para = 16.0 / m2mm
    V_norm = 78.5 / m2mm     # the trench-normal long-term loading of 78.5 mm
    if BOUND_TYPE == 'both':
        strike_bounds=(-2e-3, 2e-3)    # just a little mm. although the truth is 0
        dip_bounds=(0.0, V_norm)
        function_type=FUNCTION_TYPE
        print("Constraints to both strike and dip ")

In [ ]:
# a catalog Holocene volcanoes
volcano_file = "data/GVP_Holocene_Volcano_loc.csv" 
volcano = pd.read_csv(datadir + volcano_file, sep=",", skiprows=1, \
                      names=['id', 'lat', 'lon', 'elv'])
# Show first few rows
print(volcano.head())

region=[-87, -84, 8.5, 11.5]    # suitable region for chopping the plate interface grid file 
# region=[-86.75, -84.4, 8.75, 11.25]    # suitable region for chopping the plate interface grid file 
# region=[-88, -83, 6, 14]    # suitable region for chopping the plate interface grid file 

volcano = volcano[
    (volcano['lat'] >= region[2]) & (volcano['lat'] <= region[3]) &
    (volcano['lon'] >= region[0]) & (volcano['lon'] <= region[1])
]

In [ ]:
### Choose the slip model, all models are in m_dip direction, and m_strike = 0
### All are offseted a little along dip due to the plate shape from CK
# slipmodel = 1     # along-strike 3-stripe pattern, shallow-middle-deep
slipmodel = 2     # along-strike 2-stripe pattern with gap, shallow-deep 
# slipmodel = 3     # along-strike 1-stripe pattern, complementary of model 2, middle
# slipmodel = 4     # checkerboard pattern in strike-dip direction
# slipmodel = 5     # checkerboard pattern in x and y direction  
# slipmodel = 6     # same as model 4, but amp up to 1 m rather than V_norm=78.5 mm
# slipmodel = 7     # 2D Gaussian where the contours are circular, within range of 0 and max

V_norm = 78.5 / m2mm     # the trench-normal long-term loading of 78.5 mm

# if slipmodel == 1:
#     # A checkerboard model in m_dip, alternating between 0 and amp, and m_strike = 0
#     V_norm = 78.5 / m2mm     # the trench-normal long-term loading of 78.5 mm
#     amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
#     dx = 20e3  # grid spacing in x direction, \lamda_x = 2*dx
#     dy = 20e3  # spacing in y direction, \lamda_y = 2*dy
#     x0 = 0
#     y0 = 0
#     slip_str_gt = f"_check_x{x0/km2m:g}_y{y0/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}_ms{amp:g}"

# elif slipmodel == 2:
#     # A checkerboard model in m_dip, alternating between 0 and amp, and m_strike = 0
#     V_norm = 78.5 / m2mm     # the trench-normal long-term loading of 78.5 mm
#     amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
#     dx = 40e3  # grid spacing in x direction, \lamda_x = 2*dx
#     dy = 40e3  # spacing in y direction, \lamda_y = 2*dy
#     # x0 = 0
#     # y0 = 0
#     x0 = -20e3
#     y0 = -20e3
#     slip_str_gt = f"_check_x{x0/km2m:g}_y{y0/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}_ms{amp:g}"    

if slipmodel == 1:
    # Stripe pattern in m_dip, along-strike, 3-stripe, shallow-middle-deep; m_strike = 0
    amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    x_len = 35e3     # width of each rectangle in x direction
    dx = 25e3  # gap between rectangles
    x0 = 0e3     # x center of the pattern
    y0 = -5e3     # y center of the pattern
    # y0 = 10e3     # y center of the pattern
    # rot_deg = 45.0  # rotation angle in degrees (counter-clockwise positive)
    rot_deg = 0.0  # rotation angle in degrees (counter-clockwise positive)

    slip_str_gt = (
        # f"_stripe_x{x0/km2m:g}_y{y0/km2m:g}"
        f"_lx{x_len/km2m:g}_dx{dx/km2m:g}"
        f"_rot{rot_deg:g}_ms{amp:g}"
    )

elif slipmodel == 2:
    # Stripe pattern in m_dip, along-strike, 2-stripe, shallow-deep; m_strike = 0
    amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    x_len = 80e3     # width of each rectangle in dip direction
    y_len = 300e3   # length of each rectangle in strike direction  
    dx = 35e3  # gap between rectangles
    stripe_spacing = x_len + dx  # center-to-center distance between rectangles in dip direction
    x0 = 0     # x center of the pattern
    y0 = -45e3     # y center of the pattern
    rot_deg = 0.0  # rotation angle in degrees (counter-clockwise positive)

    # y_len = 240e3   # length of each rectangle in strike direction  
    # dx = 40e3  # gap between rectangles
    # y0 = -40e3     # y center of the pattern
    # zmin = -70e3
    # zmax = 0

    slip_str_gt = (
        f"_stripe_x{x0/km2m:g}_y{y0/km2m:g}"
        f"_lx{x_len/km2m:g}_dx{dx/km2m:g}"
        f"_rot{rot_deg:g}_ms{amp:g}"
    )

elif slipmodel == 3:
    # Stripe pattern in m_dip, along-strike, 1-stripe, middle, complementary of model 2; m_strike = 0
    amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    x_len = 40e3     # width of each rectangle in dip direction
    y_len = 300e3   # length of each rectangle in strike direction  
    dx = 100e3  # gap between rectangles
    stripe_spacing = x_len + dx  # center-to-center distance between rectangles in dip direction
    x0 = 0     # x center of the pattern
    y0 = 12.5e3     # y center of the pattern
    rot_deg = 0.0  # rotation angle in degrees (counter-clockwise positive)
    
    # y_len = 240e3   # length of each rectangle in strike direction  
    # zmin = -70e3
    # zmax = 0
    
    slip_str_gt = (
        f"_stripe_x{x0/km2m:g}_y{y0/km2m:g}"
        f"_lx{x_len/km2m:g}_dx{dx/km2m:g}"
        f"_rot{rot_deg:g}_ms{amp:g}"
    )

elif slipmodel == 4:
    # Checkerboard pattern in m_dip, in strike-dip direction; m_strike = 0
    amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    dx = 35e3  # grid spacing in x direction, \lamda_x = 2*dx
    dy = 35e3  # spacing in y direction, \lamda_y = 2*dy
    x0 = -20e3  # offset along strike
    y0 = -60e3  # offset along dip
    rot_deg = 45  # 45° counterclockwise
    zmin = -70e3
    zmax = 0
    smin = -120e3
    smax = 120e3
    slip_str_gt = f"_check_x{x0/km2m:g}_y{y0/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}_rot{rot_deg:g}_ms{amp:g}"    

elif slipmodel == 5:
    # Checkerboard pattern in m_dip, in x-y (N and E) direction; m_strike = 0,
    # given the strike direction is almost 45 deg, pattern is the same along dip
    amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    dx = 30e3  # grid spacing in x direction, \lamda_x = 2*dx
    dy = 30e3  # spacing in y direction, \lamda_y = 2*dy
    x0 = 20e3  # offset along strike
    y0 = -15e3  # offset along dip
    rot_deg = 0
    zmin = -70e3
    zmax = 0
    smin = -120e3
    smax = 120e3
    slip_str_gt = f"_check_x{x0/km2m:g}_y{y0/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}_rot{rot_deg:g}_ms{amp:g}"    

elif slipmodel == 6:
    # Checkerboard pattern in m_dip, in strike-dip direction; m_strike = 0
    amp = 1   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    dx = 35e3  # grid spacing in x direction, \lamda_x = 2*dx
    dy = 35e3  # spacing in y direction, \lamda_y = 2*dy
    x0 = -20e3  # offset along strike
    y0 = -60e3  # offset along dip
    rot_deg = 45  # 45° counterclockwise
    zmin = -70e3
    zmax = 0
    smin = -120e3
    smax = 120e3
    slip_str_gt = f"_check_x{x0/km2m:g}_y{y0/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}_rot{rot_deg:g}_ms{amp:g}"    

elif slipmodel == 7:
    # Define true model, m_strike = 0, m_dip = 2D Gaussian where the contours are circular, within range of 0 and max
    amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    x0_lon, y0_lat = -85.5, 10     # center of the pattern, in lon-lat degrees
    x_rot, y_rot  = utp.LL2ckmd(x0_lon, y0_lat, lon0, lat0, rot)
    x0, y0 = 130e3, 350e3  # offset for x and y coordinates, in m
    x0, y0 = (x_rot - x0), (y_rot - y0)   # offset to match the mesh coordinates
    sigma = 10e3  # the Gaussian would be nearly 0 at +- 3*sigma
    slip_str_gt = f"_gauss_x{round(x0/km2m):g}_y{round(y0/km2m):g}_s{sigma/km2m:g}_rs{amp:g}"

# elif slipmodel == 7:
#     # Stripe pattern in m_dip, along-strike, 2-stripe, shallow-deep; m_strike = 0
#     amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
#     x_len = 80e3     # width of each rectangle in dip direction
#     y_len = 240e3   # length of each rectangle in strike direction  
#     dx = 40e3  # gap between rectangles
#     stripe_spacing = x_len + dx  # center-to-center distance between rectangles in dip direction
#     x0 = 0e3     # x center of the pattern
#     y0 = -40e3     # y center of the pattern
#     rot_deg = 0.0  # rotation angle in degrees (counter-clockwise positive)
#     zmin = -70e3
#     zmax = 0

#     slip_str_gt = (
#         f"_stripe_x{x0/km2m:g}_y{y0/km2m:g}"
#         f"_lx{x_len/km2m:g}_dx{dx/km2m:g}"
#         f"_rot{rot_deg:g}_ms{amp:g}"
#     )

slip_str_gt1 = slip_str_gt  #with no pollution

if pollute:
    if pollute_type == 'uniform':
        slip_str_gt = slip_str_gt + "_pou"
    
    elif pollute_type == 'datastd':
        slip_str_gt = slip_str_gt + "_pod"


slip_str_gt = slip_str_gt + "_test"
slip_str_gt1 = slip_str_gt1 + "_test"   #with no pollution

print("Slip model string:", slip_str_gt)    
print("Slip model string:", slip_str_gt1)    

In [ ]:
# Define the expression of the shear modulus
def mu_expression(m):
    mu = 20*(2. + np.tanh(m))
    return mu

def mu_expression1(m):
    mu = 2*20*(2. + np.tanh(m))
    return mu

# background shear modulus
mu_b = 0   # 40 GPa
mu_background = mu_expression(mu_b)

# shear modulus for the lower (subducting) plate
mu_l = 0.9730 # ~55 GPa
mu_lower = mu_expression(mu_l)

# shear modulus for the upper (overriding) plate
mu_u = -0.9730  # ~25 GPa
# mu_u = mu_b
mu_upper = mu_expression(mu_u)

# shear modulus for volcanoes
# mu_v = -0.9730  # ~25 GPa
mu_v = 0
mu_volcano = mu_expression(mu_v) 

if mu_v == 0:
    # string for the homogeneous solution
    homo_str = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}"
    # string for the contrast between slab and wedge
    sw_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
    # # smaller contrast
    # sw_str1  = f"_mul{round(mu_expression(0.5))}u{round(mu_expression(-0.5))}"
    # # value multiplied by 2
    # sw_str2  = f"_mul{round(mu_expression1(mu_l))}u{round(mu_expression1(mu_u))}"

    # string for the 3d model
    # het3d_str = "_DeShon3D"
    # het3d_str = f"_DeShon3D_{round(1.0)}"
    # # value multiplied by 4
    # het3d_str1 = f"_DeShon3D_{round(4.0)}"
    # value multiplied by 4, relative a reference
    het3d_str2 = f"_DeShon3D_ref_{round(4.0)}"

else:
    print( "The shear modulus for the upper plate mu = %.1f and lower plate mu = %.1f and volcano mu = %.1f" %(mu_upper, mu_lower, mu_volcano) )
    sw_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}v{round(mu_expression(mu_v))}"
    # string for the homogeneous solution
    homo_str = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}v{round(mu_expression(0))}"


print(homo_str)
print(sw_str)
# print(sw_str1)
# print(sw_str2)

# print(het3d_str)
# print(het3d_str1)
print(het3d_str2)


In [ ]:

# Define the reference point (center of the projection)
lon0, lat0 = -84, 7     # from Christos's email
# convert to relative locations in meters, and then rotate
rot = 45  # rotation angle in degrees, positive is CCW
# offset in x and y direction, the same as being done to the mesh in 'Kyriakopoulos2016JGR/convert_exodus_to_msh.ipynb'
x0, y0 = 130e3, 350e3  # offset for x and y coordinates, in m

# Load the fault geometry
outFileName = 'fault_geometry_' + meshname + '.txt'
loc_f = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', skiprows=lambda x: x % 3 != 1, 
                    names=['xf', 'yf', 'zf'])
loc_f = loc_f/km2m

# Compute the inverse projection
loc_f['lon'], loc_f['lat'] = utp.ckm2LLd(loc_f['xf']*km2m+x0, loc_f['yf']*km2m+y0, lon0, lat0, -rot)
print(loc_f[['lon', 'lat']].head())
# fault.head()

# Load the ground-truth slip on the fault
outFileName = 'mtrue_s_fault_' + meshname + slip_str_gt + '.txt'
print(outFileName)
mtrue_s = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                  names=['s_strike', 's_dip'])
mtrue_s['mag'] = np.sqrt(mtrue_s['s_strike']**2 + mtrue_s['s_dip']**2)
# print(mtrue_s['s_strike'].min(), mtrue_s['s_strike'].max())
print(mtrue_s['s_dip'].min(), mtrue_s['s_dip'].max())
# print(mtrue_s['mag'].max())

if type == 'locking': 
    cols_to_convert = ['s_strike', 's_dip', 'mag']
    # mtrue_s[cols_to_convert] = mtrue_s[cols_to_convert] * m2mm  # Convert m to mm
    mtrue_s[cols_to_convert] = mtrue_s[cols_to_convert] / amp    # normalized by max
    
# print(mtrue_s['s_strike'].min(), mtrue_s['s_strike'].max())
print(mtrue_s['s_dip'].min(), mtrue_s['s_dip'].max())
# print(mtrue_s['mag'].max())

In [ ]:
#coordinate rotation function
def rot_xy(x, y, rot):
    cos_rot = np.cos(np.radians(rot))
    sin_rot = np.sin(np.radians(rot))
    x_rot = x * cos_rot + y * sin_rot
    y_rot = -x * sin_rot + y * cos_rot

    return x_rot, y_rot

# Load the ground-truth displacement for each forward structure model
def read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, struc_str_for):
    outFileName = 'd_obs_' + meshname + slip_str_gt + struc_str_for + '.txt'
    print(outFileName)
    u_true = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                         names=['x', 'y', 'z', 'ux', 'uy', 'uz'])

    u_true['lon'], u_true['lat'] = df['lon'], df['lat']  # add lon and lat for merging later    
    # the resulting disp aligned with mesh, reverse rotation, back to geo
    u_true['ux'], u_true['uy'] = rot_xy(u_true['ux'], u_true['uy'], -rot) 
    # vector magnitude
    u_true['mag'] = np.sqrt(u_true['ux']**2 + u_true['uy']**2 + u_true['uz']**2)
    u_true['mag_h'] = np.sqrt(u_true['ux']**2 + u_true['uy']**2)
    u_true['mag_v'] = u_true['uz'].abs()

    # print(u_true.head())

    return u_true

def read_station_grid(datadir, resultpath, meshname):
    # load the dense grid of station coordinates
    outFileName = 'dense_grid_coordinates_' + meshname + '.txt'
    print(outFileName)
    sta_grid = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                         names=['lon', 'lat', 'x', 'y', 'z'])    
    cols_to_convert = ['x', 'y', 'z']
    sta_grid[cols_to_convert] = sta_grid[cols_to_convert] * km2m  # Convert km to m, same as mesh units

    return sta_grid

# Load the ground-truth displacement for each forward structure model
def read_gt_disp_grid(sta_grid, datadir, resultpath, meshname, slip_str_gt, struc_str_for):
    outFileName = 'd_obs_grid' + meshname + slip_str_gt + struc_str_for + '.txt'
    print(outFileName)
    u_true = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                         names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    
    u_true['lon'], u_true['lat'] = sta_grid['lon'], sta_grid['lat']  # add lon and lat for merging later    
    # the resulting disp aligned with mesh, reverse rotation, back to geo
    u_true['ux'], u_true['uy'] = rot_xy(u_true['ux'], u_true['uy'], -rot) 
    # vector magnitude
    u_true['mag'] = np.sqrt(u_true['ux']**2 + u_true['uy']**2 + u_true['uz']**2)
    u_true['mag_h'] = np.sqrt(u_true['ux']**2 + u_true['uy']**2)
    u_true['mag_v'] = u_true['uz'].abs()

    # print(u_true.head())

    return u_true

# Build the difference displacement between two models
def build_diff_disp(u_1, u_2):
    u_diff = u_1.copy()
    u_diff['ux'] = u_2['ux'] - u_1['ux']
    u_diff['uy'] = u_2['uy'] - u_1['uy']
    u_diff['uz'] = u_2['uz'] - u_1['uz']
    u_diff['mag'] = u_2['mag'] - u_1['mag']
    u_diff['mag_h'] = u_2['mag_h'] - u_1['mag_h']
    u_diff['mag_v'] = u_2['mag_v'] - u_1['mag_v']
    # u_diff.head()
    return u_diff

# A different way of constructing the vectors for plotting arrows
def build_disp_vector(u_true, scaleunit, error_e=None, error_n=None, error_v=None):
    # convert from m to mm, horizontal components
    df_obs_h_data = {
        "x": u_true['lon'],
        "y": u_true['lat'],
        "east_velocity": u_true['ux']*scaleunit,
        "north_velocity": u_true['uy']*scaleunit,
    }
    
    # Add error columns only if errors are provided
    if error_e is not None and error_n is not None:
        df_obs_h_data["east_sigma"] = error_e*scaleunit
        df_obs_h_data["north_sigma"] = error_n*scaleunit
        df_obs_h_data["correlation_EN"] = 0.0
    
    df_obs_h = pd.DataFrame(data=df_obs_h_data)

    # convert from m to mm, vertical components
    df_obs_v_data = {
        "x": u_true['lon'],
        "y": u_true['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_true['uz']*scaleunit,
    }
    
    # Add error columns only if errors are provided
    if error_v is not None:
        df_obs_v_data["east_sigma"] = error_v*scaleunit
        df_obs_v_data["north_sigma"] = error_v*scaleunit
        df_obs_v_data["correlation_EN"] = 0.0
    
    df_obs_v = pd.DataFrame(data=df_obs_v_data)
    
    return df_obs_h, df_obs_v


# def build_diff_disp_vector(u_diff, scaleunit, error_e=None, error_n=None, error_v=None):
#     # convert from m to mm, horizontal components
#     df_diff_h_data = {
#         "x": u_diff['lon'],
#         "y": u_diff['lat'],
#         "east_velocity": u_diff['ux']*scaleunit,
#         "north_velocity": u_diff['uy']*scaleunit,
#     }
    
#     # Add error columns only if errors are provided
#     if error_e is not None and error_n is not None:
#         df_diff_h_data["east_sigma"] = error_e*scaleunit
#         df_diff_h_data["north_sigma"] = error_n*scaleunit
#         df_diff_h_data["correlation_EN"] = 0.0
    
#     df_diff_h = pd.DataFrame(data=df_diff_h_data)

#     # convert from m to mm, vertical components
#     df_diff_v_data = {
#         "x": u_diff['lon'],
#         "y": u_diff['lat'],
#         "east_velocity": 0.0,
#         "north_velocity": u_diff['uz']*scaleunit,
#     }
    
#     # Add error columns only if errors are provided
#     if error_v is not None:
#         df_diff_v_data["east_sigma"] = error_v*scaleunit
#         df_diff_v_data["north_sigma"] = error_v*scaleunit
#         df_diff_v_data["correlation_EN"] = 0.0
    
#     df_diff_v = pd.DataFrame(data=df_diff_v_data)
    
#     return df_diff_h, df_diff_v


In [ ]:
def build_disp_vector_grid(u_true, scaleunit, inc):
    """
    Downsample displacement vectors from a regular grid.
    
    Parameters:
    -----------
    df : DataFrame with 'lon', 'lat' from the original dense grid
    u_true : DataFrame with 'ux', 'uy' displacements
    inc : downsampling factor (e.g., 5 means keep every 5th point)
    
    Returns:
    --------
    vectors_h : list of vectors for PyGMT
    df_obs_h : DataFrame with downsampled data
    """
    # Determine grid dimensions from the data
    # Assuming regular grid, find unique values
    unique_lons = u_true['lon'].unique()
    unique_lats = u_true['lat'].unique()
    n_lon = len(unique_lons)
    n_lat = len(unique_lats)
    
    print(f"Detected grid: {n_lat} x {n_lon}")
    
    # Reshape to 2D arrays
    ux_2d = u_true['ux'].to_numpy().reshape(n_lat, n_lon)
    uy_2d = u_true['uy'].to_numpy().reshape(n_lat, n_lon)
    lon_2d = u_true['lon'].to_numpy().reshape(n_lat, n_lon)
    lat_2d = u_true['lat'].to_numpy().reshape(n_lat, n_lon)
    
    # Downsample in 2D (keeps regular spacing)
    ux_sub = ux_2d[::inc, ::inc]
    uy_sub = uy_2d[::inc, ::inc]
    lon_sub = lon_2d[::inc, ::inc]
    lat_sub = lat_2d[::inc, ::inc]
    
    # Flatten back to 1D
    ux = ux_sub.flatten()
    uy = uy_sub.flatten()
    
    # Calculate vector angle and length
    angle = np.degrees(np.arctan2(uy, ux))
    length = np.hypot(ux, uy)
    
    print(f"Downsampled to: {ux_sub.shape[0]} x {ux_sub.shape[1]} = {len(ux)} points")
    
    # Create DataFrame for plotting
    #format for 'pygmt.Figure.velo'
    df_obs_h_velo = pd.DataFrame(
        data={
            "x": lon_sub.flatten(),
            "y": lat_sub.flatten(),
            "east_velocity": ux * scaleunit,
            "north_velocity": uy * scaleunit,
        }
    )
    
    #format for 'pygmt.Figure.plot'
    df_obs_h_plot = pd.DataFrame({
        'x': lon_sub.flatten(),
        'y': lat_sub.flatten(),
        'angle': angle,
        'length': length * scaleunit,  # convert from m to a customized unit by scaling
    })


    return df_obs_h_velo, df_obs_h_plot 


In [ ]:
def station_grid(regiongrid, grid_spacing_deg, depth_levels):
    lon_min, lon_max = regiongrid[0], regiongrid[1]  # degrees longitude
    lat_min, lat_max = regiongrid[2], regiongrid[3]    # degrees latitude

    # Create regular lat/lon meshgrid
    lon_grid = np.arange(lon_min, lon_max + grid_spacing_deg, grid_spacing_deg)
    lat_grid = np.arange(lat_min, lat_max + grid_spacing_deg, grid_spacing_deg)
    LON_GRID, LAT_GRID = np.meshgrid(lon_grid, lat_grid)

    # Flatten for processing
    lon_2d = LON_GRID.flatten()
    lat_2d = LAT_GRID.flatten()

    # Convert 2D grid to relative coordinates
    x_rot_2d, y_rot_2d = utp.LL2ckmd(lon_2d, lat_2d, lon0, lat0, rot)
    x_2d = (x_rot_2d - x0) / 1e3  # convert to km
    y_2d = (y_rot_2d - y0) / 1e3

    # Replicate the 2D grid for each depth level
    n_2d = len(x_2d)
    n_depths = len(depth_levels)
    n_total = n_2d * n_depths

    lon_dense = np.tile(lon_2d, n_depths)
    lat_dense = np.tile(lat_2d, n_depths)
    x_dense = np.tile(x_2d, n_depths)
    y_dense = np.tile(y_2d, n_depths)
    z_dense = np.repeat(depth_levels, n_2d)  # repeat each depth n_2d times

    # Create dense grid dataframe
    sta_grid = pd.DataFrame({
        'lon': lon_dense,
        'lat': lat_dense,
        'x': x_dense,
        'y': y_dense,
        'z': z_dense
    })

    return sta_grid

# Define regular lat/lon grid covering study area
regiongrid=[-88, -83, 7.5, 12.5] 

# Grid resolution - adjust as needed for your visualization requirements
grid_spacing_deg = 0.01  # ~2 km spacing at this latitude
# grid_spacing_deg = 0.1  # ~20 km spacing at this latitude

# Define depth levels (0 to -80 km with 10 km increment)
# depth_levels = np.arange(0, -80 - 10, -10)  # [0, -10, -20, ..., -80]
depth_levels = [0]
print(f"Depth levels: {depth_levels} km")

sta_grid = station_grid(regiongrid, grid_spacing_deg, depth_levels)    
# print(sta_grid.head())

In [ ]:
# read in the observation disp. for various structure models
u_true_hom = read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, homo_str)
# print(u_true_hom.head())

# build observation disp. vectors for various structure models
df_obs_h_hom, df_obs_v_hom = build_disp_vector(u_true_hom, m2mm, error_e, error_n, error_v)
# print(df_obs_h_hom.shape)
# print(df_obs_h_hom.head())

# read the locations of the dense grid 
# sta_grid = read_station_grid(datadir, resultpath, meshname)
# print(sta_grid.head())

# read disp at each grid point
u_true_hom_grid = read_gt_disp_grid(sta_grid, datadir, resultpath, meshname, slip_str_gt, homo_str)
# print(u_true_hom_grid.head())
# print(u_true_hom_grid.shape)
print(u_true_hom_grid['uz'].min()*m2mm, u_true_hom_grid['uz'].max()*m2mm )
print(u_true_hom_grid['mag_h'].min()*m2mm, u_true_hom_grid['mag_h'].max()*m2mm )

# build observation disp. vectors at each grid point, downsampled
inc = 20    #increment, or downsample rate
df_obs_h_hom_grid, _ = build_disp_vector_grid(u_true_hom_grid, m2mm, inc)
# print(df_obs_h_hom_grid[:5])


In [ ]:
# #### FOR TESTING
# #read in the observation disp. for HOM model, NO pollution 
# u_true_hom1 = read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt1, homo_str)
# df_obs_h_hom1, df_obs_v_hom1 = build_disp_vector(u_true_hom1, m2mm)

In [ ]:
u_true_sw = read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, sw_str)
df_obs_h_sw, df_obs_v_sw = build_disp_vector(u_true_sw, m2mm, error_e, error_n, error_v)
u_true_sw_grid = read_gt_disp_grid(sta_grid, datadir, resultpath, meshname, slip_str_gt, sw_str)
df_obs_h_sw_grid, _ = build_disp_vector_grid(u_true_sw_grid, m2mm, inc)

# build disp difference between various structure models and homogeneous
u_true_swdiff = build_diff_disp(u_true_hom, u_true_sw)
u_true_swdiff_grid = build_diff_disp(u_true_hom_grid, u_true_sw_grid)
# print(u_true_swdiff_grid.head())

# build differential observation disp. vectors for various structure models wrt homogeneous
df_obs_h_swdiff, df_obs_v_swdiff = build_disp_vector(u_true_swdiff, m2mm, error_e, error_n, error_v)
df_obs_h_swdiff_grid, _ = build_disp_vector_grid(u_true_swdiff_grid, m2mm, inc)

# # smaller contrast
# u_true_sw1 = read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, sw_str1)
# df_obs_h_sw1, df_obs_v_sw1 = build_disp_vector(u_true_sw1, error_e, error_n, error_v)
# df_obs_h_swdiff1, df_obs_v_swdiff1 = build_diff_disp_vector(u_true_hom, u_true_sw1, error_e, error_n, error_v)

# # value multiplied by 2
# u_true_sw2 = read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, sw_str2)
# df_obs_h_sw2, df_obs_v_sw2 = build_disp_vector(u_true_sw2, error_e, error_n, error_v)
# df_obs_h_swdiff2, df_obs_v_swdiff2 = build_diff_disp_vector(u_true_hom, u_true_sw2, error_e, error_n, error_v)

In [ ]:
# 3D DeShon model
# u_true_3d = read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, het3d_str)
# df_obs_h_3d, df_obs_v_3d = build_disp_vector(u_true_3d, error_e, error_n, error_v)
# df_obs_h_3ddiff, df_obs_v_3ddiff = build_diff_disp_vector(u_true_hom, u_true_3d, error_e, error_n, error_v)

# # value multiplied by 4
# u_true_3d1 = read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, het3d_str1)
# df_obs_h_3d1, df_obs_v_3d1 = build_disp_vector(u_true_3d1, error_e, error_n, error_v)
# df_obs_h_3ddiff1, df_obs_v_3ddiff1 = build_diff_disp_vector(u_true_hom, u_true_3d1, error_e, error_n, error_v)

# value multiplied by 4, relative to a reference
u_true_3d2 = read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, het3d_str2)
df_obs_h_3d2, df_obs_v_3d2 = build_disp_vector(u_true_3d2, m2mm, error_e, error_n, error_v)
u_true_3d2_grid = read_gt_disp_grid(sta_grid, datadir, resultpath, meshname, slip_str_gt, het3d_str2)
df_obs_h_3d2_grid, _ = build_disp_vector_grid(u_true_3d2_grid, m2mm, inc)

u_true_3d2diff = build_diff_disp(u_true_hom, u_true_3d2)
u_true_3d2diff_grid = build_diff_disp(u_true_hom_grid, u_true_3d2_grid)

df_obs_h_3d2diff, df_obs_v_3d2diff = build_disp_vector(u_true_3d2diff, m2mm, error_e, error_n, error_v)
df_obs_h_3d2diff_grid, _ = build_disp_vector_grid(u_true_3d2diff_grid, m2mm, inc)


In [ ]:
# print(sta_grid.head(5))
# print(u_true_hom_grid.head(5))   
# len(sta_grid) == len(u_true_hom_grid)
print(u_true_swdiff_grid['mag_h'].min()*m2mm, u_true_swdiff_grid['mag_h'].max()*m2mm) 
print(u_true_swdiff_grid['uz'].min()*m2mm, u_true_swdiff_grid['uz'].max()*m2mm)
print(u_true_3d2diff_grid['mag_h'].min()*m2mm, u_true_3d2diff_grid['mag_h'].max()*m2mm) 
print(u_true_3d2diff_grid['uz'].min()*m2mm, u_true_3d2diff_grid['uz'].max()*m2mm)


In [ ]:
print(u_true_swdiff['mag_h'].min()*m2mm, u_true_swdiff['mag_h'].max()*m2mm) 
print(u_true_swdiff['uz'].min()*m2mm, u_true_swdiff['uz'].max()*m2mm)
print(u_true_3d2diff['mag_h'].min()*m2mm, u_true_3d2diff['mag_h'].max()*m2mm) 
print(u_true_3d2diff['uz'].min()*m2mm, u_true_3d2diff['uz'].max()*m2mm)

In [ ]:
# fig = pygmt.Figure()
# cmaps = ["polar", "roma", "broc", "vik"]

# for i, cmap in enumerate(cmaps):
#     fig.colorbar(
#         cmap=cmap,
#         position=f"x0c/{i*1.5}c+w10c/0.4c+h",  # <-- note 'x' not 'J'
#         frame=[f"af+l'{cmap}'"],
#         # triangular_ends=True,
#     )

# fig.show()

In [ ]:
def plot_true_slip_disp_grid(mtrue_s, col2plt, scale_vec, df_obs_h_grid, u_true_grid, region, struc_str_for, flag_savefig=False, df_obs_h_noerr=None):

    slipsybsz = "c0.08c"
    # slipsybsz = "c0.05c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", 
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")
        
        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tTrue slip"])
            maxslip = mtrue_s[col2plt].max()
            maxslip = 1*amp*m2mm
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*amp*m2mm, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*m2mm, cmap=True)   # no smoothing or interpolation
            
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
        
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            # fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.1c", fill=mtrue_s[col2plt], cmap=True)
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lSlip mag.", "y+l(mm)"])

        # Synthetic horizontal displacement
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic disp."])
            # use the vertical displacement as background color
            griduv = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['uz']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
            uz_max = u_true_grid['uz'].abs().max()*m2mm     # in mm
            # print(uz_max)
            # maxdisp = max(u_true_grid['mag_h'].max()*m2mm, u_true_grid['mag_v'].max()*m2mm)     # in mm
            # maxdisp = u_true_grid['mag'].max()*m2mm     # in mm
            maxdisp = uz_max
            pygmt.makecpt(cmap='roma', series=[-maxdisp, maxdisp, maxdisp/10], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdisp], background=True, reverse=True)
            fig.grdimage(region=region, projection="M?", grid=griduv, cmap=True, nan_transparent=True, interpolation="l")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lVert. disp.", "y+l(mm)"]) #

            fig.coast(borders=None, shorelines="0.1p,black", area_thresh=20)

            # plot the contour lines of horizontal displacement vector magnitude
            # griduh = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            # fig.grdcontour(grid=griduh, annotation="+f4p, white", pen="0.25p, white") #levels=2, 2

            # scale_vec = 0.05    # 1 mm would be scaled by 'scale_vec'
            # fig.plot(region=region, projection="M?", x=df_obs_h_grid['x'], y=df_obs_h_grid['y'],
            #          direction=[df_obs_h_grid['angle'], df_obs_h_grid['length']*scale_vec],
            #          style='v0.1c+e+jb+h0+a45', pen='0.5p,black', fill='white')     # data=df_obs_h_grid,
            # fig.velo(data=lgd, pen="0.5p,black", spec="e"+str(scale_vec), vector="0.1c+a45+p0.5p,black+ea+gwhite")
            
            #plot the horizontal displacement vectors over the grid, error-free
            fig.velo(data=df_obs_h_grid, pen="0.5p,black", spec="e"+str(0.5/scale_vec)+"/0.39", 
                     vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
            #plot legends
            fig.plot(x=region[0]+0.35, y=region[-2]+0.2, style="r1/0.5", fill="white", pen=None, transparency=30, )
            lgd = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.3, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgd, pen="0.5p,black", spec="e0.5/0.39", vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
            fig.text(text=str(int(scale_vec))+" mm", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")

        # direct comparison with error-free displacement vectors at real station locations 
        if df_obs_h_noerr is not None:
            with fig.set_panel(panel=[0, 2]):
                fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tCompare at real stations"])
                #plot the horizontal displacement vectors over the grid, error-free
                fig.velo(data=df_obs_h_grid, pen="0.5p,black", spec="e"+str(0.5/scale_vec)+"/0.39", 
                        vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
                #plot the horizontal displacement vectors over real station, error-free
                fig.velo(data=df_obs_h_noerr, pen="0.5p,magenta", spec="e"+str(0.5/scale_vec)+"/0.39", 
                                    vector="0.1c+a45+p0.1p,magenta+ea+gmagenta+h0") 
                # plot legends
                fig.plot(x=region[0]+0.35, y=region[-2]+0.2, style="r1/0.5", fill="white", pen=None, transparency=30, )
                lgd = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.3, "east_velocity": [1], "north_velocity": [0]})
                fig.velo(data=lgd, pen="0.5p,black", spec="e0.5/0.39", vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
                fig.text(text=str(int(scale_vec))+" mm", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")                

    fig.show()

    if flag_savefig: 
        fig.savefig(figpath + meshname + struc_str_for + str(slipmodel) + "_trueslip.pdf")

uh_max = u_true_hom_grid['mag_h'].max()*m2mm
n = np.ceil(uh_max / 10)  # get the n where scale_vec=5*n would make the ratio between disp and scale_vec is less than 2
scale_vec = 5*n    # vector length are plotted relative to scale_vec*length in original unit, here mm 
print(uh_max, scale_vec)

# #plot the true slip model, and resulting disp vectors over a regular grid under the respective structure models, compare them at real stations without error 
# plot_true_slip_disp_grid(mtrue_s, 's_dip', scale_vec, df_obs_h_hom_grid, u_true_hom_grid, region, homo_str, flag_savefig=False, df_obs_h_noerr=df_obs_h_hom1)

#plot the true slip model, and resulting disp vectors over a regular grid under the respective structure models
plot_true_slip_disp_grid(mtrue_s, 's_dip', scale_vec, df_obs_h_hom_grid, u_true_hom_grid, region, homo_str, flag_savefig=False)
plot_true_slip_disp_grid(mtrue_s, 's_dip', scale_vec, df_obs_h_sw_grid, u_true_sw_grid, region, sw_str, flag_savefig=False)
plot_true_slip_disp_grid(mtrue_s, 's_dip', scale_vec, df_obs_h_3d2_grid, u_true_3d2_grid, region, het3d_str2, flag_savefig=False)

In [ ]:
#plot the differential disp vectors between models over a regular grid
scale_vec = 5
plot_true_slip_disp_grid(mtrue_s, 's_dip', scale_vec, df_obs_h_swdiff_grid, u_true_swdiff_grid, region, sw_str, flag_savefig=False)
plot_true_slip_disp_grid(mtrue_s, 's_dip', scale_vec, df_obs_h_3d2diff_grid, u_true_3d2diff_grid, region, het3d_str2, flag_savefig=False)


In [ ]:
def compare_true_disp_wo_errors(scale_vec, df_obs_h, df_obs_v, df_obs_h_noerr, df_obs_v_noerr, region):

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=2, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", 
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")
 
        # Synthetic horizontal displacement
        with fig.set_panel(panel=[0, 0]):

            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic horiz. disp."], borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)   #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            # fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            #Synthetic displacement
            fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e"+str(0.5/scale_vec)+"/0.39", 
                     vector="0.1c+a45+p1p,black+ea+gblack+h0")
            fig.velo(data=df_obs_h_noerr, pen="0.5p,red", spec="e"+str(0.5/scale_vec)+"/0.39", 
                                vector="0.1c+a45+p1p,red+ea+gred+h0")            
            #plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                                  "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.4, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdobs, pen="0.5p,red", spec="e0.5/0.39", vector="0.1c+a45+p1p,red+ea+gred+h0")
            #plot texts sigma
            fig.text(text="Stations", x=region[0]+0.45, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs H ± 1\u03C3", x=region[0]+0.45, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Error-free", x=region[0]+0.45, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text=str(int(scale_vec))+"±"+str(int(scale_vec/5))+" mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # Synthetic vertical displacement
        with fig.set_panel(panel=[0, 1]):
            fig.coast(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic vert. disp."], borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            # fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
                    
            #Synthetic displacement
            fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e"+str(0.5/scale_vec)+"/0.39",
                     vector="0.1c+a45+p1p,black+ea+gblack+h0")
            fig.velo(data=df_obs_v_noerr, pen="0.5p,red", spec="e"+str(0.5/scale_vec)+"/0.39", 
                                vector="0.1c+a45+p1p,red+ea+gred+h0") 
            #plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.35, "y": region[-2]+0.45, "east_velocity": [0], "north_velocity": [1],
                                  "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.3, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdobs, pen="0.5p,red", spec="e0.5/0.39", vector="0.1c+a45+p1p,red+ea+gred+h0")
            #plot texts
            fig.text(text="Stations", x=region[0]+0.45, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs V ± 1\u03C3", x=region[0]+0.45, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Error-free", x=region[0]+0.45, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text=str(int(scale_vec))+"±"+str(int(scale_vec/5))+" mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML") 


    fig.show()


uh_max = np.hypot(df_obs_h_hom['east_velocity'].to_numpy(), df_obs_h_hom['north_velocity'].to_numpy()).max()
n = np.ceil(uh_max / 10)  # get the n where scale_vec=5*n would make the ratio between disp and scale_vec is less than 2
scale_vec = np.round(5*n)    # vector length are plotted relative to scale_vec*length in original unit, here mm 
print(uh_max, scale_vec)

# plot the true slip model, and resulting disp vectors under the respective structure models
# compare_true_disp_wo_errors(scale_vec, df_obs_h_hom, df_obs_v_hom, df_obs_h_hom1, df_obs_v_hom1, region)


In [ ]:
def plot_true_slip_disp(mtrue_s, col2plt, scale_vec, df_obs_h, df_obs_v, region, struc_str_for, flag_savefig=False, u_true_grid=None):

    slipsybsz = "c0.08c"
    # slipsybsz = "c0.05c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", 
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")
        
        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tTrue slip"])
            maxslip = mtrue_s[col2plt].max()
            maxslip = 1*amp*m2mm
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*amp*m2mm, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*m2mm, cmap=True)   # no smoothing or interpolation
            
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
        
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            # fig.plot(x=-84.869, y=10.8536, style="s0.15c", fill="red", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lSlip mag.", "y+l(mm)"])

        # Synthetic horizontal displacement
        with fig.set_panel(panel=[0, 1]):
            fig.coast(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic horiz. disp."], borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)   #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            # fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            if u_true_grid is not None:
                griduh = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
                # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
                # data_bmedian = pygmt.blockmedian(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01))
                # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
                maxdisp = u_true_grid['mag_h'].max()*m2mm     # in mm
                # maxdisp = max(u_true_grid['mag_h'].max()*m2mm, u_true_grid['mag_v'].max()*m2mm)     # in mm
                # maxdisp = u_true_grid['mag'].max()*m2mm     # in mm
                pygmt.makecpt(cmap=colormap, series=[0, maxdisp, maxdisp/20], background=True, reverse=True)
                # pygmt.makecpt(cmap=colormap, series=[0, maxdisp], background=True, reverse=True)
                fig.grdimage(grid=griduh, cmap=True, nan_transparent=True, interpolation="l")
                with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                    fig.colorbar(frame=["af", "x+lHoriz. disp. mag.", "y+l(mm)"]) #
                # fig.grdcontour(grid=griduh, levels=5, annotation="5+f4p, black", pen="0.25p, black") #

            #Synthetic displacement
            fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e"+str(0.5/scale_vec)+"/0.39", 
                     vector="0.1c+a45+p1p,black+ea+gblack+h0")
            #plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.5, "east_velocity": [1], "north_velocity": [0],
                                  "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            #plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs H", x=region[0]+0.6, y=region[-2]+0.5, font="8p,Helvetica,black", justify="ML")
            fig.text(text=str(int(scale_vec))+"±"+str(int(scale_vec/5))+" mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # Synthetic vertical displacement
        with fig.set_panel(panel=[0, 2]):
            fig.coast(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic vert. disp."], borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            # fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            if u_true_grid is not None:
                griduz = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_v']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
                # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
                # data_bmedian = pygmt.blockmedian(x=sta_grid['lon'], y=sta_grid['lat'], z=u_true_grid['mag_v']*m2mm, region=region, spacing=(0.01, 0.01))
                # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
                maxdisp = u_true_grid['mag_v'].max()*m2mm     # in mm
                pygmt.makecpt(cmap=colormap, series=[0, maxdisp, maxdisp/20], background=True, reverse=True)
                # pygmt.makecpt(cmap=colormap, series=[0, maxdisp], background=True, reverse=True)
                fig.grdimage(grid=griduz, cmap=True, nan_transparent=True, interpolation="l")
                with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                    fig.colorbar(frame=["af", "x+lVert. disp. mag.", "y+l(mm)"]) # 
                # fig.grdcontour(grid=griduz, levels=5, annotation="5+f4p, black", pen="0.25p, black") #
                    
            #Synthetic displacement
            fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e"+str(0.5/scale_vec)+"/0.39",
                     vector="0.1c+a45+p1p,black+ea+gblack+h0")
            #plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.3, "east_velocity": [0], "north_velocity": [1],
                                  "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            #plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs V", x=region[0]+0.6, y=region[-2]+0.5, font="8p,Helvetica,black", justify="ML")
            fig.text(text=str(int(scale_vec))+"±"+str(int(scale_vec/5))+" mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML") 


    fig.show()

    if flag_savefig: 
        fig.savefig(figpath + meshname + struc_str_for + str(slipmodel) + "_trueslip.pdf")


uh_max = np.hypot(df_obs_h_hom['east_velocity'].to_numpy(), df_obs_h_hom['north_velocity'].to_numpy()).max()
n = np.ceil(uh_max / 10)  # get the n where scale_vec=5*n would make the ratio between disp and scale_vec is less than 2
scale_vec = np.round(5*n)    # vector length are plotted relative to scale_vec*length in original unit, here mm 
print(uh_max, scale_vec)

# plot the true slip model, and resulting disp vectors under the respective structure models
plot_true_slip_disp(mtrue_s, 's_dip', scale_vec, df_obs_h_hom, df_obs_v_hom, region, homo_str, flag_savefig=False, u_true_grid=u_true_hom_grid)
# plot_true_slip_disp(mtrue_s, 's_dip', scale_vec, df_obs_h_hom, df_obs_v_hom, region, homo_str, flag_savefig=False)

plot_true_slip_disp(mtrue_s, 's_dip', scale_vec, df_obs_h_sw, df_obs_v_sw, region, sw_str, flag_savefig=False, u_true_grid=u_true_sw_grid)
plot_true_slip_disp(mtrue_s, 's_dip', scale_vec, df_obs_h_3d2, df_obs_v_3d2, region, het3d_str2, flag_savefig=False, u_true_grid=u_true_3d2_grid)

# plot_true_slip_disp(mtrue_s, 's_dip', u_true_sw_grid, df_obs_h_swdiff, df_obs_v_swdiff, region, sw_str, flag_savefig=False)
# plot_true_slip_disp(mtrue_s, 's_dip', df_obs_h_swdiff1, df_obs_v_swdiff1, region, sw_str, flag_savefig=False)
# plot_true_slip_disp(mtrue_s, 's_dip', df_obs_h_swdiff2, df_obs_v_swdiff2, region, sw_str, flag_savefig=False)

# plot_true_slip_disp(mtrue_s, 's_dip', df_obs_h_hom, df_obs_v_hom, region, homo_str, flag_savefig=False)
# plot_true_slip_disp(mtrue_s, 's_dip', df_obs_h_3ddiff, df_obs_v_3ddiff, region, het3d_str, flag_savefig=False)
# plot_true_slip_disp(mtrue_s, 's_dip', df_obs_h_3ddiff1, df_obs_v_3ddiff1, region, het3d_str1, flag_savefig=False)
# plot_true_slip_disp(mtrue_s, 's_dip', u_true_3d_grid, df_obs_h_3ddiff2, df_obs_v_3ddiff2, region, het3d_str2, flag_savefig=False)

In [ ]:
def plot_true_slip_disp_diff(mtrue_s, col2plt, scale_vec, maxdiff, u_true_diff_grid, df_obs_diff_h, df_obs_diff_v, region, struc_str_for, flag_savefig=False):

    slipsybsz = "c0.08c"
    # slipsybsz = "c0.05c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", 
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")
        
        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tTrue slip"])
            maxslip = mtrue_s[col2plt].max()
            maxslip = 1*amp*m2mm
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            # fig.plot(x=[lonlt, lonrt, lonrb, lonlb, lonlt], y=[latlt, latrt, latrb, latlb, latlt], pen=True, fill="green", transparency=50)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=-mtrue_s[col2plt], region=region, spacing=(0.02, 0.02)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=mtrue_s[col2plt], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*amp*m2mm, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*m2mm, cmap=True)   # no smoothing or interpolation
            
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
        
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.1c", fill=mtrue_s[col2plt], cmap=True)
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lSlip mag.", "y+l(mm)"])

        # Synthetic horizontal displacement
        with fig.set_panel(panel=[0, 1]):
            fig.coast(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic horiz. disp."], borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)   #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            # fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            # grid = pygmt.xyz2grd(x=sta_grid['lon'], y=sta_grid['lat'], z=u_true_diff_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            data_bmedian = pygmt.blockmedian(x=sta_grid['lon'], y=sta_grid['lat'], z=u_true_diff_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01))
            grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
            # maxdiff = u_true_diff_grid['mag_h'].abs().max()*m2mm     # in mm
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff], background=True, reverse=True)
            fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation="l")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lDiff. in horiz. disp. mag.", "y+l(mm)"]) #

            #Synthetic displacement
            fig.velo(data=df_obs_diff_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e"+str(0.5/scale_vec)+"/0.39", 
                     vector="0.1c+a45+p1p,black+ea+gblack+h0")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.5, "east_velocity": [1], "north_velocity": [0],
                                  "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs H", x=region[0]+0.6, y=region[-2]+0.5, font="8p,Helvetica,black", justify="ML")
            fig.text(text=str(int(scale_vec))+"±"+str(int(scale_vec/5))+" mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML") 

        # Synthetic vertical displacement
        with fig.set_panel(panel=[0, 2]):
            fig.coast(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic vert. disp."], borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            # grid = pygmt.xyz2grd(x=sta_grid['lon'], y=sta_grid['lat'], z=u_true_diff_grid['uz']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            data_bmedian = pygmt.blockmedian(x=sta_grid['lon'], y=sta_grid['lat'], z=u_true_diff_grid['uz']*m2mm, region=region, spacing=(0.01, 0.01))
            grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
            # maxdiff = u_true_diff_grid['uz'].abs().max()*m2mm     # in mm
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff], background=True, reverse=True)
            fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation="l")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lDiff. in vert. disp. mag.", "y+l(mm)"]) #
                fig.colorbar(frame=["af", "x+lDiff. in vert. disp.", "y+l(mm)"]) #

            #Synthetic displacement
            fig.velo(data=df_obs_diff_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e"+str(0.5/scale_vec)+"/0.39",
                     vector="0.1c+a45+p1p,black+ea+gblack+h0")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.3, "east_velocity": [0], "north_velocity": [1],
                                  "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs V", x=region[0]+0.6, y=region[-2]+0.5, font="8p,Helvetica,black", justify="ML")
            fig.text(text=str(int(scale_vec))+"±"+str(int(scale_vec/5))+" mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML") 


    fig.show()

    if flag_savefig: 
        fig.savefig(figpath + meshname + struc_str_for + str(slipmodel) + "_trueslipdiff.pdf")

maxdiff = round(max(u_true_swdiff_grid['mag_h'].abs().max(), u_true_swdiff_grid['mag_v'].abs().max(), 
                    u_true_3d2diff_grid['mag_h'].abs().max(), u_true_3d2diff_grid['mag_v'].abs().max()) * m2mm)
print("Overall max diff in disp.: ", maxdiff)
scale_vec = 5
plot_true_slip_disp_diff(mtrue_s, 's_dip', scale_vec, maxdiff, u_true_swdiff_grid, df_obs_h_swdiff, df_obs_v_swdiff, region, sw_str, flag_savefig=False)
plot_true_slip_disp_diff(mtrue_s, 's_dip', scale_vec, maxdiff, u_true_3d2diff_grid, df_obs_h_3d2diff, df_obs_v_3d2diff, region, het3d_str2, flag_savefig=False)
